# Clase 06 · Implementación: embeddings y búsqueda semántica

Sebastián Cruz Cardona   
Código: 240220222010

Partimos del clasificador y la consulta de solicitudes de las clases anteriores; hoy añadimos una forma de buscar información en un corpus. Esta demo recupera pasajes, pero todavía no genera respuestas con ellos.

Usaremos cinco textos **ficticios**. Ejecuta las celdas en orden en Google Colab. La primera ejecución descarga un modelo de embeddings; no se necesita clave de API ni GPU.

## 1. Preparación
Instalamos la biblioteca para generar embeddings. En una sesión nueva de Colab se ejecuta esta celda una vez.

In [1]:
%pip -q install sentence-transformers

## 2. Corpus
Cada pasaje conserva un identificador y su texto.

In [2]:
corpus = [
    {
        "id": "P1",
        "texto": "Constitución Política de Colombia. Artículo 17: Se prohiben la esclavitud, la servidumbre y la trata de seres humanos en todas sus formas."
    },
    {
        "id": "P2",
        "texto": "El sistema funciona como un chat. El usuario debe buscar la interfaz de entrada y formular su pregunta. Si requiere consultar por el estado de un proceso, incluir el código, conformado por P- seguido de 11 dígitos."
    },
    {
        "id": "P3",
        "texto": "La Corte Constitucional, en la sentencia su642-98, afirma que el derecho fundamental al libre desarrollo de la personalidad protege la capacidad de las personas para definir, en forma autónoma, las opciones vitales que habrán de guiar el curso de su existencia."
    },
    {
        "id": "P4",
        "texto": "La actualización del estado de un proceso judicial es un trámite diferente al de consulta y está fuera del alcance del sistema."
    },
    {
        "id": "P5",
        "texto": "Código Sustantivo del Trabajo. Artículo 28. Utilidades y Pérdidas. El trabajador puede participar de las utilidades o beneficios de su {empleador}, pero nunca asumir sus riesgos o pérdidas."
    },
]

for pasaje in corpus:
    print(f"{pasaje['id']}: {pasaje['texto']}")

P1: Constitución Política de Colombia. Artículo 17: Se prohiben la esclavitud, la servidumbre y la trata de seres humanos en todas sus formas.
P2: El sistema funciona como un chat. El usuario debe buscar la interfaz de entrada y formular su pregunta. Si requiere consultar por el estado de un proceso, incluir el código, conformado por P- seguido de 11 dígitos.
P3: La Corte Constitucional, en la sentencia su642-98, afirma que el derecho fundamental al libre desarrollo de la personalidad protege la capacidad de las personas para definir, en forma autónoma, las opciones vitales que habrán de guiar el curso de su existencia.
P4: La actualización del estado de un proceso judicial es un trámite diferente al de consulta y está fuera del alcance del sistema.
P5: Código Sustantivo del Trabajo. Artículo 28. Utilidades y Pérdidas. El trabajador puede participar de las utilidades o beneficios de su {empleador}, pero nunca asumir sus riesgos o pérdidas.


## 3. Convertimos el corpus en vectores
Usamos un [modelo multilingüe de Sentence Transformers](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2). `normalize_embeddings=True` deja los vectores con longitud unitaria; así, su producto punto corresponde a la similitud coseno. Los cinco vectores se calculan **una sola vez**, antes de escribir la pregunta.

Mostramos la forma de la matriz y solo los primeros valores de un vector. Los números no se interpretan uno por uno como palabras o reglas.

In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer

NOMBRE_MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
modelo = SentenceTransformer(NOMBRE_MODELO)
textos = [pasaje["texto"] for pasaje in corpus]
vectores_corpus = modelo.encode(textos, convert_to_numpy=True, normalize_embeddings=True)

print("Modelo:", NOMBRE_MODELO)
print("Forma de la matriz (pasajes, dimensiones):", vectores_corpus.shape)
print("Primeros 8 valores de P1:", np.round(vectores_corpus[0, :8], 3))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Forma de la matriz (pasajes, dimensiones): (5, 384)
Primeros 8 valores de P1: [-0.053  0.03   0.006  0.076  0.148  0.004  0.021 -0.053]


## 4. Una búsqueda literal breve
El término `quiebra` aparece en la pregunta que haremos, pero no en nuestro corpus. Una coincidencia literal con esa palabra no encontrará ningún pasaje. Esto prepara la comparación con la búsqueda semántica.

In [5]:
termino = "quiebra"
coincidencias = [p["id"] for p in corpus if termino in p["texto"].casefold()]
print(f"Pasajes que contienen literalmente '{termino}': {coincidencias}")

Pasajes que contienen literalmente 'quiebra': []


## 5. Escribe aquí la pregunta del usuario
**Esta es la celda donde ingresamos el texto del usuario.** En Colab se puede editar el campo `CONSULTA` del formulario o cambiar directamente el texto entre comillas y ejecutar de nuevo. Predicción antes de ejecutar: ¿esperamos ver P5 entre los primeros resultados?

In [20]:
CONSULTA = "¿El trabajador debe responder económicamente por la quiebra de la empresa de su empleador?" #@param {type:"string"}
K = 3 #@param {type:"integer"}
print("Consulta recibida:", CONSULTA)

Consulta recibida: ¿El trabajador debe responder económicamente por la quiebra de la empresa de su empleador?


## 6. Buscamos los pasajes más cercanos
Convertimos la consulta con el **mismo modelo**, calculamos la similitud coseno y ordenamos los resultados. `K` indica cuántos pasajes mostramos. El puntaje permite ordenar; no representa un porcentaje de certeza.

In [21]:
def buscar(consulta, k=3):
    if not consulta.strip():
        raise ValueError("Escribe una consulta antes de buscar.")
    if not 1 <= k <= len(corpus):
        raise ValueError(f"K debe estar entre 1 y {len(corpus)}.")

    vector_consulta = modelo.encode(consulta, convert_to_numpy=True, normalize_embeddings=True)
    puntajes = vectores_corpus @ vector_consulta
    indices = np.argsort(-puntajes)[:k]
    return [(corpus[int(i)]["id"], float(puntajes[i]), corpus[int(i)]["texto"]) for i in indices]

resultados = buscar(CONSULTA, K)
for puesto, (identificador, similitud, texto) in enumerate(resultados, 1):
    print(f"{puesto}. {identificador} | similitud={similitud:.3f} | {texto}")

assert all(resultados[i][1] >= resultados[i + 1][1] for i in range(len(resultados) - 1))

1. P5 | similitud=0.631 | Código Sustantivo del Trabajo. Artículo 28. Utilidades y Pérdidas. El trabajador puede participar de las utilidades o beneficios de su {empleador}, pero nunca asumir sus riesgos o pérdidas.
2. P4 | similitud=0.110 | La actualización del estado de un proceso judicial es un trámite diferente al de consulta y está fuera del alcance del sistema.
3. P2 | similitud=0.101 | El sistema funciona como un chat. El usuario debe buscar la interfaz de entrada y formular su pregunta. Si requiere consultar por el estado de un proceso, incluir el código, conformado por P- seguido de 11 dígitos.


## 7. Ejecución de pruebas
Ejecutamos 5 pruebas.
1. Paráfrasis -> Se espera P5.
2. Coincidencia exacta -> Se espera P1.
3. Soporte -> Se espera P2.
4. Consulta por porceso que corpus no puede resolver -> Se espera P2 o P4.
5. PRegunta fuera de covertura -> No se sabe qué sugerirá.

In [26]:
preguntas = [
    "¿El trabajador debe responder económicamente por la quiebra de la empresa de su empleador?",
    "¿Según la Constitución se prohibe o permite la esclavitud en Colombia?",
    "¿Cómo se hace para consultar el estado de un proceso?",
    "¿Cuál es el estado actual de P-20260123457?",
    "¿Cómo solicito una audiencia con el juez?"
]

for i in range(len(preguntas)):
    print(f"\nPregunta {i+1}: {preguntas[i]}")
    for identificador, similitud, texto in buscar(preguntas[i], k=3):
        print(f"  {identificador} | {similitud:.3f} | {texto}")


Pregunta 1: ¿El trabajador debe responder económicamente por la quiebra de la empresa de su empleador?
  P5 | 0.631 | Código Sustantivo del Trabajo. Artículo 28. Utilidades y Pérdidas. El trabajador puede participar de las utilidades o beneficios de su {empleador}, pero nunca asumir sus riesgos o pérdidas.
  P4 | 0.110 | La actualización del estado de un proceso judicial es un trámite diferente al de consulta y está fuera del alcance del sistema.
  P2 | 0.101 | El sistema funciona como un chat. El usuario debe buscar la interfaz de entrada y formular su pregunta. Si requiere consultar por el estado de un proceso, incluir el código, conformado por P- seguido de 11 dígitos.

Pregunta 2: ¿Según la Constitución se prohibe o permite la esclavitud en Colombia?
  P1 | 0.853 | Constitución Política de Colombia. Artículo 17: Se prohiben la esclavitud, la servidumbre y la trata de seres humanos en todas sus formas.
  P3 | 0.275 | La Corte Constitucional, en la sentencia su642-98, afirma que el 

## Criterios de aceptación para el trabajo
1. Sustituir `corpus` por unos pocos pasajes **autorizados y verificables** del proyecto; conservar identificadores, procedencia y fecha de consulta.
2. Ejecuten una paráfrasis, una coincidencia exacta y una pregunta fuera de cobertura.
3. Registren en la bitácora el modelo, `K`, los pasajes recuperados y un fallo observado.

**Nota:** La preparación e indexación de documentos completos corresponde a la Clase 07.

Documentación: [Sentence Transformers — Semantic Search](https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html).